# Infrared Solar Modules: Fault vs Normal CNN with 2-Class Softmax (TensorFlow/Keras, no XPU)

This notebook builds a **2-class categorical classifier** on the **Infrared Solar Modules** dataset by collapsing the original classes into:

- **normal** = `No-Anomaly`
- **fault** = every other original anomaly class

The goal is to keep the model **small, fast, and practical** while also returning an explicit **softmax probability distribution** such as:

- `fault = 0.80`
- `normal = 0.20`

That means this notebook uses a **2-unit softmax output** instead of a single-logit binary head.

## Model used

```text
Input
Conv(32) -> BN -> ReLU -> MaxPool
Conv(64) -> BN -> ReLU -> MaxPool
Conv(96) -> BN -> ReLU -> MaxPool
GAP
Dense(32) -> ReLU
Dropout(0.2)
Dense(2) -> Softmax
```

## Notebook flow

1. Load the dataset from `module_metadata.json`
2. Run very light EDA
3. Preprocess images and collapse labels to 2 classes
4. Split into train / validation / test
5. Train a compact CNN with **2-class softmax**
6. Inspect predicted probabilities
7. Evaluate on the test set
8. Summarize the reasoning behind the solution


In [ ]:

# If needed on a fresh environment, uncomment:
# !pip install -q tensorflow scikit-learn pandas pillow matplotlib seaborn


In [ ]:
import os
import json
import random
import platform
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

os.environ["ONEDNN_VERBOSE"] = "none"
os.environ["DNNL_VERBOSE"] = "0"
warnings.filterwarnings("ignore", category=UserWarning)

print("TensorFlow version:", tf.__version__)


## Configuration

Set `DATASET_DIR` to the root folder of the extracted **Infrared Solar Modules** dataset.

This notebook is aligned to the dataset format defined by **`module_metadata.json`**:

```text
DATASET_DIR/
    module_metadata.json
    images/
        0.jpg
        1.jpg
        2.jpg
        ...
```

Each metadata entry contains at least:

- `image_filepath`
- `anomaly_class`

The binary experiment is then built by mapping:

- `No-Anomaly` -> `normal`
- everything else -> `fault`


In [ ]:
# -----------------------------
# Configuration
# -----------------------------
DATASET_DIR = Path("classwork/solar/dataset/2020-02-14_InfraredSolarModules/InfraredSolarModules")  # <- change if needed
MODULE_METADATA_PATH = DATASET_DIR / "module_metadata.json"

IMG_HEIGHT = 24
IMG_WIDTH = 40
CHANNELS = 1

BATCH_SIZE = 32
EPOCHS = 10
SEED = 42

CLASS_NAMES = ["normal", "fault"]   # label 0 = normal, label 1 = fault
NUM_CLASSES = len(CLASS_NAMES)

AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

NORMAL_CLASS_NAMES = {
    "no-anomaly", "no_anomaly", "no anomaly", "normal", "noanomaly"
}

print("Runtime    : standard TensorFlow / Keras (no explicit XPU selection)")

In [ ]:
def normalize_class_name(name: str) -> str:
    return name.strip().lower().replace("_", "-")

def binary_label_from_class(original_class: str) -> int:
    norm = normalize_class_name(original_class)
    return 0 if norm in NORMAL_CLASS_NAMES else 1  # 0 = normal, 1 = fault

def load_dataset_from_metadata(dataset_dir: Path, metadata_path: Path) -> pd.DataFrame:
    if not dataset_dir.exists():
        raise FileNotFoundError(
            f"DATASET_DIR does not exist: {dataset_dir}\n"
            "Update DATASET_DIR to the extracted dataset folder."
        )
    if not metadata_path.exists():
        raise FileNotFoundError(
            f"module_metadata.json not found: {metadata_path}\n"
            "This notebook expects the RaptorMaps-style metadata file."
        )

    payload = json.loads(metadata_path.read_text(encoding="utf-8"))
    rows = []

    for module_id, record in payload.items():
        rel_path = record.get("image_filepath")
        original_class = record.get("anomaly_class")

        if rel_path is None or original_class is None:
            continue

        abs_path = dataset_dir / rel_path
        if not abs_path.exists():
            continue

        binary_label = binary_label_from_class(original_class)
        rows.append(
            {
                "module_id": str(module_id),
                "filepath": str(abs_path),
                "relative_filepath": rel_path,
                "original_class": original_class,
                "binary_label": binary_label,
                "binary_class": CLASS_NAMES[binary_label],
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No valid image rows could be built from module_metadata.json.")
    return df

df = load_dataset_from_metadata(DATASET_DIR, MODULE_METADATA_PATH)
df.head()


## Quick EDA

In [ ]:
# Basic dataset metrics
n_images = len(df)
n_original_classes = df["original_class"].nunique()
binary_counts = df["binary_class"].value_counts().reindex(CLASS_NAMES, fill_value=0)
original_counts = df["original_class"].value_counts().sort_values(ascending=False)

# Read image sizes (core metric only)
sizes = []
for fp in df["filepath"]:
    with Image.open(fp) as im:
        sizes.append(im.size)  # (width, height)

size_series = pd.Series(sizes)
unique_sizes = size_series.value_counts()

print(f"Total images         : {n_images:,}")
print(f"Original class count : {n_original_classes}")
print("Binary class counts  :")
display(binary_counts.to_frame("count"))
print("Most common image sizes (width, height):")
display(unique_sizes.head(10).rename("count").to_frame())


In [ ]:

# EDA Plot 1: original class distribution + binary class distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(
    x=original_counts.index,
    y=original_counts.values,
    ax=axes[0],
)
axes[0].set_title("Original Class Distribution")
axes[0].set_xlabel("Original class")
axes[0].set_ylabel("Image count")
axes[0].tick_params(axis="x", rotation=75)

sns.barplot(
    x=binary_counts.index,
    y=binary_counts.values,
    ax=axes[1],
)
axes[1].set_title("Binary Class Distribution")
axes[1].set_xlabel("Binary class")
axes[1].set_ylabel("Image count")

plt.tight_layout()
plt.show()


In [ ]:

# EDA Plot 2: a few sample thermal images
def show_random_samples(dataframe, n_per_class=4):
    classes = ["normal", "fault"]
    fig, axes = plt.subplots(len(classes), n_per_class, figsize=(3*n_per_class, 5))

    if len(classes) == 1:
        axes = np.array([axes])

    for row_idx, cls in enumerate(classes):
        subset = dataframe[dataframe["binary_class"] == cls].sample(
            n=min(n_per_class, len(dataframe[dataframe["binary_class"] == cls])),
            random_state=SEED
        )
        for col_idx, (_, row) in enumerate(subset.iterrows()):
            img = Image.open(row["filepath"]).convert("L")
            axes[row_idx, col_idx].imshow(img, cmap="inferno")
            axes[row_idx, col_idx].set_title(f"{cls}\n{row['original_class']}")
            axes[row_idx, col_idx].axis("off")

    plt.suptitle("Random Samples from Each Binary Class", y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

show_random_samples(df, n_per_class=4)


## Split the data

We use a **stratified** split so the dataset remains stable across train, validation, and test.

To better preserve the underlying source distribution, the split is stratified by the **original anomaly class** first, not only by the collapsed binary label.

- Train: 70%
- Validation: 15%
- Test: 15%


In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["original_class"],
    random_state=SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["original_class"],
    random_state=SEED,
)

print("Train size:", len(train_df))
print("Val size  :", len(val_df))
print("Test size :", len(test_df))

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = split_df["binary_class"].value_counts(normalize=True).reindex(CLASS_NAMES, fill_value=0)
    print(f"\n{name} binary ratio:")
    print(counts)


## Preprocessing and augmentation

### Preprocessing
- load as grayscale
- resize to `(24, 40)`
- convert to float
- scale pixels to `[0, 1]`

### Labels
We keep integer labels:

- `0 = normal`
- `1 = fault`

The model then learns a **2-class softmax** over these two categories.

### Training augmentation
- left-right flip
- up-down flip
- slight brightness jitter

These are intentionally **light**. Thermal images are not the place for aggressive visual circus tricks.


In [ ]:
def decode_resize_normalize(filepath, label):
    image_bytes = tf.io.read_file(filepath)
    image = tf.io.decode_image(image_bytes, channels=1, expand_animations=False)
    image.set_shape([None, None, 1])
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = tf.cast(image, tf.float32) / 255.0
    return image, tf.cast(label, tf.int32)

data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomFlip("vertical"),
        layers.RandomBrightness(factor=0.08),
    ],
    name="light_augmentation",
)

def make_dataset(dataframe, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (dataframe["filepath"].values, dataframe["binary_label"].values.astype("int32"))
    )
    ds = ds.map(decode_resize_normalize, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df, training=False)
test_ds = make_dataset(test_df, training=False)


## Build the compact CNN

The convolutional backbone stays compact, but the output head is now:

- `Dense(2, activation="softmax")`

This makes the model return an explicit categorical distribution:

- `P(normal | image)`
- `P(fault | image)`


In [ ]:
def conv_bn_relu(x, filters, kernel_size=3):
    x = layers.Conv2D(filters, kernel_size, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def build_model(input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS), num_classes=NUM_CLASSES):
    inputs = keras.Input(shape=input_shape)

    x = conv_bn_relu(inputs, 32)
    x = layers.MaxPooling2D(pool_size=2)(x)

    x = conv_bn_relu(x, 64)
    x = layers.MaxPooling2D(pool_size=2)(x)

    x = conv_bn_relu(x, 96)
    x = layers.MaxPooling2D(pool_size=2)(x)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="class_probs")(x)

    model = keras.Model(inputs, outputs, name="fault_vs_normal_softmax_cnn")
    return model

model = build_model()
model.summary()


## Compile and train

Training setup:
- **Optimizer**: AdamW
- **Loss**: Sparse categorical cross-entropy
- **Monitored validation metric**: validation accuracy
- **Early stopping**: restores the best weights

Because the output is now a **2-class softmax**, the loss switches from BCE-with-logits to **sparse categorical cross-entropy**.


This version uses standard TensorFlow execution without an explicit XPU device-selection block.

In [ ]:
optimizer = keras.optimizers.AdamW(
    learning_rate=1e-3,
    weight_decay=1e-4
)

model.compile(
        optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        ],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        "best_fault_vs_normal_softmax_cnn.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)


In [ ]:

# Compact training curves
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["loss"], label="train")
axes[0].plot(history_df["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["auc"], label="train")
axes[1].plot(history_df["val_auc"], label="val")
axes[1].set_title("AUC")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## Inspect probability outputs

Since the model uses **softmax**, each prediction already contains a full 2-class probability distribution:

- `normal probability`
- `fault probability`

These two probabilities sum to 1.


In [ ]:
def predict_probabilities(model, ds):
    probs = model.predict(ds, verbose=0)
    labels = np.concatenate([y.numpy().ravel() for _, y in ds], axis=0).astype(int)
    pred_idx = np.argmax(probs, axis=1)
    return labels, probs, pred_idx

y_val, p_val, y_val_pred = predict_probabilities(model, val_ds)

preview_df = pd.DataFrame(
    {
        "normal_probability": p_val[:, 0],
        "fault_probability": p_val[:, 1],
        "predicted_class": [CLASS_NAMES[i] for i in y_val_pred],
        "true_class": [CLASS_NAMES[i] for i in y_val],
    }
)

display(preview_df.head(10))

print("Validation Accuracy :", accuracy_score(y_val, y_val_pred))
print("Validation Precision:", precision_score(y_val, y_val_pred, zero_division=0))
print("Validation Recall   :", recall_score(y_val, y_val_pred, zero_division=0))
print("Validation F1       :", f1_score(y_val, y_val_pred, zero_division=0))
print("Validation ROC-AUC  :", roc_auc_score(y_val, p_val[:, 1]))


## Final evaluation on the test set


In [ ]:
y_test, p_test, y_pred = predict_probabilities(model, test_ds)

print("Test Accuracy :", accuracy_score(y_test, y_pred))
print("Test Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Test Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("Test F1       :", f1_score(y_test, y_pred, zero_division=0))
print("Test ROC-AUC  :", roc_auc_score(y_test, p_test[:, 1]))

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, zero_division=0))

test_preview_df = pd.DataFrame(
    {
        "fault_probability": p_test[:, 1],
        "normal_probability": p_test[:, 0],
        "predicted_class": [CLASS_NAMES[i] for i in y_pred],
        "true_class": [CLASS_NAMES[i] for i in y_test],
    }
).sort_values("fault_probability", ascending=False)

display(test_preview_df.head(10))


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Test Confusion Matrix")
plt.tight_layout()
plt.show()



## Save the trained model (optional)

Uncomment if you want an explicit export at the end.

```python
# model.save("fault_vs_normal_cnn_final.keras")
```


# Explanation and reasoning behind the solution

## 1. This is still binary classification, but encoded as 2-class softmax
The original dataset contains multiple fault categories plus one healthy class.

For this notebook, the labels are collapsed into:

- **normal** = `No-Anomaly`
- **fault** = all other classes

That means the *task* is still binary in meaning, but the *model output* is now **categorical with 2 classes**.

Why do this?

Because a 2-class softmax directly returns both probabilities:

- `P(normal | image)`
- `P(fault | image)`

So you can get an output like:

- `fault = 0.80`
- `normal = 0.20`

This is exactly what you asked for.

---

## 2. Why switch from one logit to softmax
A single-logit binary head is perfectly valid, but it naturally gives you one main probability, usually the probability of the positive class.

A **2-class softmax** is often easier to explain in reports and demos because it explicitly shows the model's belief over both categories.

So this notebook uses:

```text
Dense(2) -> Softmax
```

and trains with:

- **SparseCategoricalCrossentropy**

instead of:

- binary cross-entropy with logits

---

## 3. Why the notebook now reads `module_metadata.json`
The Infrared Solar Modules dataset is provided through a metadata file where each module record contains fields such as:

- `image_filepath`
- `anomaly_class`

That means the dataset is not naturally organized as one folder per class.  
So the notebook now loads the images by following the metadata file and then derives the 2-class target from `anomaly_class`.

This is the correct alignment for the dataset format.

---

## 4. Why stratify by original anomaly class
Even though the final task is only `normal` vs `fault`, the source dataset still contains many different anomaly subclasses such as cracking, diode faults, shadowing, soiling, and so on.

If we stratify only on the final binary label, some rare original fault types could be distributed badly across splits.

So the split is stratified by the **original class first**, which preserves the source distribution more faithfully.

---

## 5. Why use the same XPU runtime selection style as `logmel_cnn_v2_2.py`
The notebook now mirrors the runtime-selection logic used in `model_training/logmel_cnn_v2_2.py`:

1. try CUDA GPU first
2. if not available, try Intel XPU through `intel_extension_for_tensorflow`
3. otherwise fall back to CPU

It also performs a small **smoke-test matrix multiplication** on the selected device.

This is useful because it makes the notebook behave more like your existing training stack and gives you a practical path to Intel XPU acceleration when the environment supports it.

---

## 6. Why keep the CNN compact
The thermal images in this dataset are small, so a giant backbone would often add computation faster than it adds useful signal.

This compact CNN is a good balance between:

- accuracy
- speed
- memory usage
- deployment practicality

Architecture:

```text
Conv(32) -> BN -> ReLU -> MaxPool
Conv(64) -> BN -> ReLU -> MaxPool
Conv(96) -> BN -> ReLU -> MaxPool
GAP
Dense(32) -> ReLU
Dropout(0.2)
Dense(2) -> Softmax
```

It stays light, but still has enough capacity to separate normal thermal patterns from abnormal ones.

---

## 7. Why GAP is still a good choice
**Global Average Pooling (GAP)** replaces each final feature map with one average value.

This gives you:

- fewer parameters
- lower overfitting risk
- lower compute cost
- cleaner deployment behavior

In plain language, GAP asks each learned feature map:

> “Across the whole image, how strongly did this pattern appear?”

That is a very sensible question for fault-vs-normal thermal classification.

---

## 8. Why only light augmentation
Thermal imagery is more delicate than natural-image photography.  
Aggressive augmentation can distort the thermal meaning of the sample.

So this notebook keeps augmentation modest:

- flips
- slight brightness jitter

That gives extra variety without turning the data into nonsense soup.

---

## 9. Which metrics matter most
Even though the model is trained with softmax, the practical evaluation question is still:

> how well does it detect faulty modules?

So the key metrics remain:

- **Accuracy**
- **Precision**
- **Recall**
- **F1-score**
- **ROC-AUC**
- **Confusion matrix**

And because the model outputs two probabilities, you can also inspect uncertain examples more easily.

---

## 10. Bottom line
This notebook now does three aligned things:

1. loads the dataset the way it is actually provided through `module_metadata.json`
2. predicts with a **2-class softmax** so you get explicit `fault` and `normal` probabilities
3. uses the same **XPU/GPU/CPU runtime selection pattern** as your `logmel_cnn_v2_2.py`

So the result is cleaner, more consistent with your training stack, and more demo-friendly.


---

## 12. Why this version removes the XPU-specific runtime block
This notebook keeps the **same softmax formulation** but removes the explicit device-selection logic.

That makes it easier to run in environments where you just want standard TensorFlow behavior:

- CPU-only machines
- generic TensorFlow installs
- notebook environments without Intel XPU / ITEX configured

In other words, this version is the simpler, cleaner sibling notebook:
same dataset logic, same 2-class softmax idea, fewer runtime-specific moving parts.
